In [ ]:
repository_filter: list[str] = []
highlight_cycles: str = "true"

In [ ]:
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.graph_objects as go
import numpy as np
import warnings

warnings.simplefilter("ignore")

df = dt.read_csv("../samples/package_quality_metrics.csv")
df = qu.filter_repos(df, repository_filter)
show_cycles = highlight_cycles.lower() == "true"

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    fig = go.Figure()

    # Shaded Zone of Pain (bottom-left: high stability, low abstractness)
    fig.add_shape(
        type="rect",
        x0=0, y0=0, x1=0.5, y1=0.5,
        fillcolor="rgba(239,83,80,0.10)",
        line=dict(width=0),
        layer="below",
    )
    fig.add_annotation(
        x=0.15, y=0.15, text="Zone of Pain",
        showarrow=False, font=dict(size=12, color="#EF5350"),
    )

    # Shaded Zone of Uselessness (top-right: high abstractness, high instability)
    fig.add_shape(
        type="rect",
        x0=0.5, y0=0.5, x1=1, y1=1,
        fillcolor="rgba(255,224,130,0.15)",
        line=dict(width=0),
        layer="below",
    )
    fig.add_annotation(
        x=0.85, y=0.85, text="Zone of Uselessness",
        showarrow=False, font=dict(size=12, color="#F9A825"),
    )

    # Main sequence diagonal line (A + I = 1)
    fig.add_shape(
        type="line",
        x0=0, y0=1, x1=1, y1=0,
        line=dict(color="#888", width=2, dash="dash"),
    )
    fig.add_annotation(
        x=0.65, y=0.42, text="Main Sequence",
        showarrow=False, font=dict(size=10, color="#888"),
        textangle=-45,
    )

    # Separate cycle vs non-cycle packages
    in_cycle = df["inCycle"].astype(str).str.lower() == "true"
    normal = df[~in_cycle]
    cycled = df[in_cycle]

    # Non-cycle packages as circles
    if len(normal) > 0:
        fig.add_trace(
            go.Scatter(
                x=normal["instability"],
                y=normal["abstractness"],
                mode="markers",
                marker=dict(
                    size=10,
                    color=normal["distanceFromMainSequence"],
                    colorscale=qu.HEALTH_COLORSCALE[::-1],
                    cmin=0,
                    cmax=1,
                    colorbar=dict(title="Distance from<br>Main Sequence"),
                    symbol="circle",
                ),
                text=normal["packageName"],
                hovertemplate=(
                    "<b>%{text}</b><br>"
                    "Instability: %{x:.2f}<br>"
                    "Abstractness: %{y:.2f}<br>"
                    "Distance: %{marker.color:.2f}"
                    "<extra></extra>"
                ),
                name="Package",
            )
        )

    # Cycle packages as stars
    if show_cycles and len(cycled) > 0:
        fig.add_trace(
            go.Scatter(
                x=cycled["instability"],
                y=cycled["abstractness"],
                mode="markers",
                marker=dict(
                    size=14,
                    color=cycled["distanceFromMainSequence"],
                    colorscale=qu.HEALTH_COLORSCALE[::-1],
                    cmin=0,
                    cmax=1,
                    symbol="star",
                    line=dict(width=1, color="#EF5350"),
                ),
                text=cycled["packageName"],
                hovertemplate=(
                    "<b>%{text}</b> (in cycle)<br>"
                    "Instability: %{x:.2f}<br>"
                    "Abstractness: %{y:.2f}<br>"
                    "Distance: %{marker.color:.2f}"
                    "<extra></extra>"
                ),
                name="In Cycle",
            )
        )

    fig.update_layout(
        title="Architectural Stability Scatter (Instability vs Abstractness)",
        xaxis=dict(title="Instability (I)", range=[-0.05, 1.05]),
        yaxis=dict(title="Abstractness (A)", range=[-0.05, 1.05]),
        height=700,
        width=800,
        plot_bgcolor="white",
        margin=dict(l=60, r=60, t=60, b=60),
    )
fig.show()